In [ ]:
import sys, os
#sys.path.append(os.path.abspath('..'))
from utils.model_loader import get_model_fits
import numpy as np
import pandas as pd
import re
from sklearn.metrics import mean_squared_error
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
data_dir = "datasets/friedman"
results_dir = "results/regression/single_layer_H16/tanh/friedman"
results_dir_corr = "results/regression/single_layer/tanh/friedman_correlated"

model_names = ["DHS_H16_L2", "DHS_H32_L2"]#["Gaussian", "RHS", "DHS", "DST"]

fits, fits_corr = {}, {}
for fname in sorted(f for f in os.listdir(data_dir) if f.endswith(".npz") and "_N50_" not in f):
    config = fname.replace(".npz", "")
    fit = get_model_fits(config=config, results_dir=results_dir,
                         models=model_names, include_prior=False)
    if fit:
        fits[config] = fit

# data_dir_corr = "datasets/friedman_correlated"
# for fname in sorted(f for f in os.listdir(data_dir_corr) if f.endswith(".npz")):
#     config = fname.replace(".npz", "")
#     fit = get_model_fits(config=config, results_dir=results_dir_corr,
#                          models=model_names, include_prior=False)
#     if fit:
#         fits_corr[config] = fit


In [ ]:
test_fit = fits['Friedman_N200_p10_sigma1.00_seed7']

In [ ]:
def forward_pass_tanh_deep(X, W1, W_internals, W_L, b_hidden, b_out):
    """
    Multi-layer tanh forward pass for a single posterior sample.

    X:           (N, P)
    W1:          (P, H)          — input-to-first-hidden weights
    W_internals: list of (H, H)  — one matrix per internal (hidden-to-hidden) layer
    W_L:         (H, O)          — last-hidden-to-output weights
    b_hidden:    (num_layers, H) — bias for each hidden layer (input + internal)
    b_out:       (O,) or scalar  — output bias

    Returns (N, O) predictions.
    """
    h = np.tanh(X @ W1 + b_hidden[0])
    for i, W_int in enumerate(W_internals):
        h = np.tanh(h @ W_int + b_hidden[i + 1])
    return h @ W_L + np.atleast_1d(b_out).reshape(1, -1)

In [ ]:
config = "Friedman_N200_p10_sigma1.00_seed7"

data = np.load(f"datasets/friedman/{config}.npz")
X_test = data["X_test"]   # (N_test, P)

preds = {}
for model_name, entry in test_fit.items():
    fit = entry["posterior"]
    W_1         = fit.stan_variable("W_1")          # (draws, P, H)
    hidden_bias = fit.stan_variable("hidden_bias")  # (draws, L, H)
    W_L         = fit.stan_variable("W_L")          # (draws, H, 1)
    output_bias = fit.stan_variable("output_bias")  # (draws, 1)

    L = hidden_bias.shape[1]
    W_internal_all = fit.stan_variable("W_internal") if L > 1 else None

    n_draws = W_1.shape[0]
    out = np.zeros((n_draws, X_test.shape[0]))
    for s in range(n_draws):
        W_int_list = [W_internal_all[s, l] for l in range(L - 1)] if L > 1 else []
        out[s] = forward_pass_tanh_deep(
            X_test,
            W_1[s],
            W_int_list,
            W_L[s],
            hidden_bias[s],
            output_bias[s],
        ).ravel()
    preds[model_name] = out  # (n_draws, N_test)

{k: v.shape for k, v in preds.items()}


In [ ]:
y_test = data["y_test"].ravel()  # (N_test,)

def crps_empirical(samples, y):
    """CRPS via E|X-y| - 0.5*E|X-X'| using Monte Carlo draws."""
    e_xy = np.mean(np.abs(samples - y))
    e_xx = np.mean(np.abs(samples[:, None] - samples[None, :]))
    return e_xy - 0.5 * e_xx

results = {}
for model_name, draws in preds.items():   # draws: (n_draws, N_test)
    post_mean = draws.mean(axis=0)
    rmse = np.sqrt(np.mean((post_mean - y_test) ** 2))

    crps = np.mean([crps_empirical(draws[:, i], y_test[i]) for i in range(len(y_test))])

    lo, hi = np.percentile(draws, [5, 95], axis=0)
    coverage_90 = np.mean((y_test >= lo) & (y_test <= hi))

    results[model_name] = {"RMSE": rmse, "CRPS": crps, "90% coverage": coverage_90}

pd.DataFrame(results).T.round(3)


In [ ]:
for model_name, entry in test_fit.items():
    fit = entry["posterior"]
    draws = preds[model_name]
    sigma = fit.stan_variable("sigma")

    print(f"\n--- {model_name} ---")
    print(f"  sigma:          mean={sigma.mean():.3f}, std={sigma.std():.3f}")
    print(f"  pred std across test pts (mean over draws): {draws.std(axis=1).mean():.4f}")
    print(f"  pred range per draw (mean):                 {(draws.max(axis=1) - draws.min(axis=1)).mean():.4f}")

    #summary = fit.summary()
    #rhat = summary["R_hat"].dropna()
    #ess  = summary["N_Eff"].dropna()
    #print(f"  R-hat:          max={rhat.max():.3f}, n params > 1.01: {(rhat > 1.01).sum()}")
    #print(f"  ESS:            min={ess.min():.0f}, median={ess.median():.0f}")
